# Large Language Model Application  
## 以 Hugging Face Transformers Pipeline API 進行命名實體識別  
### Named Entity Recognition, NER

本教材示範如何使用 Hugging Face `transformers` 函式庫中的 `pipeline` API 執行命名實體識別。

命名實體識別，英文為 Named Entity Recognition，簡稱 NER，是自然語言處理中的一項重要任務。  
它的目標是從一段文字中自動找出具有特定意義的實體，例如：

- 人名，例如：Tim Cook
- 組織名稱，例如：Apple
- 地點名稱，例如：Taipei
- 其他專有名詞，例如：iPhone、Hugging Face

在實務應用中，NER 常用於新聞分析、客服系統、知識抽取、法規文件分析與搜尋系統前處理。

## 1. Pipeline API 簡介

Hugging Face Transformers 提供了 `pipeline` API，讓使用者可以用非常簡潔的方式呼叫預訓練模型。

一般而言，使用 pipeline 只需要指定任務名稱，例如：

```python
pipeline("ner")

In [2]:
# 匯入 Hugging Face Transformers 的 pipeline
from transformers import pipeline

c:\Users\mapd\OneDrive - ispan.com.tw\桌面\2_教學研發\05_Natural Language Processing\3_Code4Test\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. 建立 NER Pipeline

接下來建立一個命名實體識別 pipeline。

此處使用公開的英文 NER 預訓練模型：

`dbmdz/bert-large-cased-finetuned-conll03-english`

這個模型常用於英文命名實體識別，可辨識常見的實體類型，例如：

| 實體標籤 | 意義 |
|---|---|
| PER | Person，人名 |
| ORG | Organization，組織 |
| LOC | Location，地點 |
| MISC | Miscellaneous，其他專有名詞 |

參數 `aggregation_strategy="simple"` 的作用是將被切成多個 token 的實體合併成較完整的詞組。  
例如模型可能將 `Hugging Face` 拆成多個片段，透過 aggregation strategy 可以讓輸出結果更容易閱讀。

In [3]:
# 建立命名實體識別 pipeline
ner_pipeline = pipeline(
    task="ner",
    model="dbmdz/bert-large-cased-finetuned-conll03-english",
    aggregation_strategy="simple"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 422.30it/s, Materializing param=classifier.weight]                                      
BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 3. 準備示範文本

以下範例使用公開、客觀且去識別化的產品與科技新聞風格文字。

文字內容描述某公司於某城市發表產品，適合用來展示 NER 如何辨識：

- 公司或組織名稱
- 地點名稱
- 產品或專有名詞
- 人名

In [4]:
# 設定示範文本
text = (
    "Apple introduced a new iPhone model at a technology event in Taipei. "
    "During the presentation, Tim Cook discussed the company's investment in artificial intelligence."
)

# 顯示原始文本
print(text)

Apple introduced a new iPhone model at a technology event in Taipei. During the presentation, Tim Cook discussed the company's investment in artificial intelligence.


## 4. 執行命名實體識別

現在將文字傳入 NER pipeline。

pipeline 會回傳一個 list，其中每一筆結果代表模型偵測到的一個實體。  
每個實體通常包含以下資訊：

| 欄位 | 說明 |
|---|---|
| entity_group | 實體類別 |
| score | 模型信心分數 |
| word | 被辨識出的文字 |
| start | 實體在原文中的起始位置 |
| end | 實體在原文中的結束位置 |

In [5]:
# 執行命名實體識別
ner_results = ner_pipeline(text)

# 顯示原始預測結果
ner_results

[{'entity_group': 'ORG',
  'score': np.float32(0.99874675),
  'word': 'Apple',
  'start': 0,
  'end': 5},
 {'entity_group': 'MISC',
  'score': np.float32(0.99436104),
  'word': 'iPhone',
  'start': 23,
  'end': 29},
 {'entity_group': 'LOC',
  'score': np.float32(0.99961805),
  'word': 'Taipei',
  'start': 61,
  'end': 67},
 {'entity_group': 'PER',
  'score': np.float32(0.99972093),
  'word': 'Tim Cook',
  'start': 94,
  'end': 102}]

## 5. 格式化輸出結果

為了讓結果更適合教學展示，以下將模型輸出整理成較容易閱讀的格式。

特別注意：

- `score` 代表模型對該預測結果的信心分數
- 分數越接近 1，代表模型越有信心
- 實體類型由模型自動判斷，並非由使用者提供候選標籤

In [6]:
# 格式化輸出 NER 預測結果
for item in ner_results:
    entity_text = item["word"]                     # 取得實體文字
    entity_type = item["entity_group"]             # 取得實體類型
    confidence = item["score"]                     # 取得信心分數
    start_pos = item["start"]                      # 取得起始位置
    end_pos = item["end"]                          # 取得結束位置

    # 使用格式化字串輸出結果
    print(f"實體文字：{entity_text}")
    print(f"實體類型：{entity_type}")
    print(f"信心分數：{confidence:.4f}")
    print(f"文字位置：{start_pos} 到 {end_pos}")
    print("-" * 40)

實體文字：Apple
實體類型：ORG
信心分數：0.9987
文字位置：0 到 5
----------------------------------------
實體文字：iPhone
實體類型：MISC
信心分數：0.9944
文字位置：23 到 29
----------------------------------------
實體文字：Taipei
實體類型：LOC
信心分數：0.9996
文字位置：61 到 67
----------------------------------------
實體文字：Tim Cook
實體類型：PER
信心分數：0.9997
文字位置：94 到 102
----------------------------------------
